# 03 — Dashboard Visuals
> **Superstore Sales Analytics | Future Interns DS Task 1**

This notebook prototypes all **interactive Plotly charts** that power `dashboard.py`.
Run each cell here to preview the chart, then the same code is imported by the Streamlit app.

---
### Charts built here
1. KPI metric cards
2. Monthly Sales & Profit line chart
3. Sub-category profit waterfall / bar
4. Region performance grouped bar
5. Discount vs Profit scatter
6. Segment pie / donut
7. Category sales treemap
8. State-level choropleth map

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

# ── Shared theme ──────────────────────────────────────────
TEMPLATE   = 'plotly_white'
COLOR_SEQ  = px.colors.qualitative.Set2
GREEN      = '#2ca02c'
RED        = '#d62728'
BLUE       = '#1f77b4'
FONT_TITLE = dict(size=16, color='#222')

df = pd.read_csv('../dataset/processed/superstore_clean.csv', parse_dates=['Order Date', 'Ship Date'])
df['YearMonth'] = df['Order Date'].dt.to_period('M')

print(f'Loaded {len(df):,} rows ✓')

Loaded 9,994 rows ✓


## 1. KPI Cards (Indicator traces)

In [2]:
total_sales   = df['Sales'].sum()
total_profit  = df['Profit'].sum()
margin        = total_profit / total_sales * 100
n_customers   = df['Customer ID'].nunique()
n_orders      = df['Order ID'].nunique()

fig = go.Figure()

kpis = [
    ('Total Revenue',   f'${total_sales/1e6:.2f}M', BLUE),
    ('Total Profit',    f'${total_profit/1e3:.1f}K', GREEN),
    ('Profit Margin',   f'{margin:.1f}%',            GREEN),
    ('Unique Customers', f'{n_customers:,}',          '#7f7f7f'),
    ('Total Orders',    f'{n_orders:,}',             '#7f7f7f'),
]

for i, (label, value, color) in enumerate(kpis):
    fig.add_trace(go.Indicator(
        mode='number',
        value=None,
        number={'prefix': ''},
        title={'text': f'<b>{value}</b><br><span style="font-size:13px;color:{color}">{label}</span>'},
        domain={'x': [i / 5, (i + 1) / 5], 'y': [0, 1]}
    ))

fig.update_layout(
    template=TEMPLATE,
    height=150,
    margin=dict(t=30, b=10, l=10, r=10),
    title=dict(text='Dashboard KPIs', font=FONT_TITLE)
)
fig.show()

## 2. Monthly Sales & Profit Dual-Axis Chart

In [3]:
monthly = (
    df.groupby('YearMonth')[['Sales', 'Profit']]
    .sum()
    .reset_index()
)
monthly['YearMonth'] = monthly['YearMonth'].dt.to_timestamp()

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Scatter(
        x=monthly['YearMonth'], y=monthly['Sales'],
        name='Sales', line=dict(color=BLUE, width=2),
        fill='tozeroy', fillcolor='rgba(31,119,180,0.08)'
    ),
    secondary_y=False
)

fig.add_trace(
    go.Bar(
        x=monthly['YearMonth'], y=monthly['Profit'],
        name='Profit',
        marker_color=[GREEN if v >= 0 else RED for v in monthly['Profit']],
        opacity=0.7
    ),
    secondary_y=True
)

fig.update_layout(
    title=dict(text='Monthly Sales & Profit (2014–2017)', font=FONT_TITLE),
    template=TEMPLATE,
    hovermode='x unified',
    legend=dict(x=0.01, y=0.99),
    height=420
)
fig.update_yaxes(title_text='Sales ($)', tickprefix='$', secondary_y=False)
fig.update_yaxes(title_text='Profit ($)', tickprefix='$', secondary_y=True)
fig.show()

## 3. Sub-Category Profit Bar (sorted)

In [4]:
subcat = (
    df.groupby('Sub-Category')['Profit']
    .sum()
    .reset_index()
    .sort_values('Profit')
)
subcat['Color'] = subcat['Profit'].apply(lambda x: RED if x < 0 else GREEN)

fig = go.Figure(go.Bar(
    x=subcat['Profit'],
    y=subcat['Sub-Category'],
    orientation='h',
    marker_color=subcat['Color'],
    text=subcat['Profit'].apply(lambda x: f'${x:,.0f}'),
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Profit: $%{x:,.0f}<extra></extra>'
))

fig.add_vline(x=0, line_color='black', line_width=1)
fig.update_layout(
    title=dict(text='Total Profit by Sub-Category — Red = Loss', font=FONT_TITLE),
    template=TEMPLATE,
    xaxis=dict(title='Total Profit ($)', tickprefix='$'),
    height=520,
    margin=dict(l=120)
)
fig.show()

## 4. Region Performance Grouped Bar

In [5]:
region = (
    df.groupby('Region')[['Sales', 'Profit']]
    .sum()
    .reset_index()
    .sort_values('Sales', ascending=False)
)

fig = go.Figure()
fig.add_trace(go.Bar(name='Sales',  x=region['Region'], y=region['Sales'],  marker_color=BLUE))
fig.add_trace(go.Bar(name='Profit', x=region['Region'], y=region['Profit'], marker_color=GREEN))

fig.update_layout(
    barmode='group',
    title=dict(text='Sales & Profit by Region', font=FONT_TITLE),
    template=TEMPLATE,
    yaxis=dict(title='Amount ($)', tickprefix='$'),
    legend=dict(x=0.75, y=0.99),
    height=400
)
fig.show()

## 5. Discount vs Profit Scatter

In [6]:
sample = df.sample(n=3000, random_state=42)

fig = px.scatter(
    sample,
    x='Discount', y='Profit',
    color='Category',
    opacity=0.45,
    color_discrete_sequence=COLOR_SEQ,
    hover_data=['Sub-Category', 'Region', 'Sales'],
    title='Discount vs Profit per Transaction',
    template=TEMPLATE,
    labels={'Discount': 'Discount Applied', 'Profit': 'Profit ($)'}
)

# Reference lines
fig.add_hline(y=0,   line_dash='dash', line_color='black',  annotation_text='Break-even')
fig.add_vline(x=0.2, line_dash='dot',  line_color='orange', annotation_text='20% threshold')

fig.update_layout(height=450)
fig.show()

## 6. Customer Segment Donut

In [7]:
seg_sales = df.groupby('Segment')['Sales'].sum().reset_index()

fig = go.Figure(go.Pie(
    labels=seg_sales['Segment'],
    values=seg_sales['Sales'],
    hole=0.45,
    marker_colors=COLOR_SEQ,
    hovertemplate='<b>%{label}</b><br>Sales: $%{value:,.0f}<br>Share: %{percent}<extra></extra>'
))

fig.update_layout(
    title=dict(text='Revenue Share by Customer Segment', font=FONT_TITLE),
    template=TEMPLATE,
    annotations=[dict(text='Segments', x=0.5, y=0.5, font_size=13, showarrow=False)],
    height=380
)
fig.show()

## 7. Category × Sub-Category Treemap

In [8]:
tree_data = (
    df.groupby(['Category', 'Sub-Category'])
    .agg(Sales=('Sales', 'sum'), Profit=('Profit', 'sum'))
    .reset_index()
)

fig = px.treemap(
    tree_data,
    path=['Category', 'Sub-Category'],
    values='Sales',
    color='Profit',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    title='Sales Treemap — Colour = Profit (Green=High, Red=Loss)',
    template=TEMPLATE,
    hover_data={'Sales': ':$,.0f', 'Profit': ':$,.0f'}
)
fig.update_layout(height=500, margin=dict(t=50, l=10, r=10, b=10))
fig.show()

## 8. State-Level Choropleth Map

In [9]:
# US state name → abbreviation lookup
state_abbr = {
    'Alabama':'AL','Alaska':'AK','Arizona':'AZ','Arkansas':'AR','California':'CA',
    'Colorado':'CO','Connecticut':'CT','Delaware':'DE','Florida':'FL','Georgia':'GA',
    'Hawaii':'HI','Idaho':'ID','Illinois':'IL','Indiana':'IN','Iowa':'IA',
    'Kansas':'KS','Kentucky':'KY','Louisiana':'LA','Maine':'ME','Maryland':'MD',
    'Massachusetts':'MA','Michigan':'MI','Minnesota':'MN','Mississippi':'MS',
    'Missouri':'MO','Montana':'MT','Nebraska':'NE','Nevada':'NV','New Hampshire':'NH',
    'New Jersey':'NJ','New Mexico':'NM','New York':'NY','North Carolina':'NC',
    'North Dakota':'ND','Ohio':'OH','Oklahoma':'OK','Oregon':'OR','Pennsylvania':'PA',
    'Rhode Island':'RI','South Carolina':'SC','South Dakota':'SD','Tennessee':'TN',
    'Texas':'TX','Utah':'UT','Vermont':'VT','Virginia':'VA','Washington':'WA',
    'West Virginia':'WV','Wisconsin':'WI','Wyoming':'WY','District of Columbia':'DC'
}

state_profit = (
    df.groupby('State')
    .agg(Sales=('Sales', 'sum'), Profit=('Profit', 'sum'), Orders=('Order ID', 'nunique'))
    .reset_index()
)
state_profit['Code'] = state_profit['State'].map(state_abbr)

fig = px.choropleth(
    state_profit,
    locations='Code',
    locationmode='USA-states',
    color='Profit',
    scope='usa',
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    hover_name='State',
    hover_data={'Sales': ':$,.0f', 'Profit': ':$,.0f', 'Orders': True, 'Code': False},
    title='Profit by US State — Red = Loss-Making States',
    template=TEMPLATE
)
fig.update_layout(height=450, margin=dict(t=50, b=0, l=0, r=0))
fig.show()

## 9. Helper — Export all charts as static PNGs
*(Optional: requires `kaleido` — `pip install kaleido`)*

In [10]:
# Uncomment to export all charts as static images for the README / report
# import os
# FIGURES = '../reports/figures'
# os.makedirs(FIGURES, exist_ok=True)

# charts = {
#     'monthly_sales_trend': fig_monthly,    # assign each fig to a variable above first
#     'subcategory_profit':  fig_subcat,
#     'regional_performance': fig_region,
#     'state_choropleth':    fig_map,
# }
# for name, fig in charts.items():
#     fig.write_image(f'{FIGURES}/{name}.png', width=1200, height=600)
#     print(f'Saved {name}.png')

print('Uncomment the block above to export static PNGs (needs kaleido).')

Uncomment the block above to export static PNGs (needs kaleido).


---
## What's Next
All charts above are production-ready for `dashboard.py`.  
In the Streamlit app, wrap each chart in:
```python
st.plotly_chart(fig, use_container_width=True)
```
and add `st.sidebar` filters for **Year**, **Region**, and **Category** to make it interactive.